# Tarefa 3 — Primeira árvore de decisão

**Equipe 18** — Bruno Alves Ribeiro de Souza, Igor da Silva Alves Correa, Pedro Henrique Alexandre da Silva, Samuel de Oliveira Santos, Victoria Agatha Rodrigues Fagundes, Wendel Henrique da Silva Rocha
**Scrum Master da tarefa:** Wendel Henrique da Silva Rocha

> A Tarefa 2 fez a ficha e o rótulo. Este notebook **não refaz** nenhum dos dois: lê o treino e a validação já cortados.
>
> **O arquivo de teste não é aberto aqui.** Ele fica fechado até a Tarefa 5. A escolha de `max_depth` e `min_samples_leaf` usa **só a validação**.

| | Artefato | Origem / destino |
|---|---|---|
| **Entra** | Ficha de baseline, dataset rotulado e corte temporal (`treino.csv`, `validacao.csv`) | Tarefa 2 — os mesmos |
| **Sai** | Árvore treinada só no treino, com critério e profundidade registrados | Tarefa 4 mexe nesta árvore, não no rótulo |
| **Sai** | Regras da árvore em texto e matriz 3×3 | Tarefa 4 |
| **Sai** | Árvore de contraste (com rota ou RTT absoluto) e a frase do que ela aprendeu | Prova de que o modelo oficial não segue o atalho |

## 0. Bibliotecas, caminhos e parâmetros

Bibliotecas externas: **pandas**, **numpy** e **scikit-learn (>= 1.3)**. A versão 1.3 é necessária porque a árvore do scikit-learn só aceita valores vazios (`NaN`) nas features a partir dela; assim o RTT ausente continua vazio, sem ser trocado por 0.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.tree import DecisionTreeClassifier, export_text

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)
_v = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
assert _v >= (1, 3), "Atualize o scikit-learn (>= 1.3): a árvore precisa aceitar NaN nas features."

pandas 3.0.6 | numpy 2.5.4 | scikit-learn 1.9.1


In [2]:
# A raiz do repositório é localizada sozinha: o notebook funciona de qualquer subpasta (ex.: notebook/arvore/).
CURRENT_DIR = Path.cwd().resolve()
ROOT = next((c for c in (CURRENT_DIR, *CURRENT_DIR.parents) if (c / "data_ripe_atlas" / "processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Não encontrei data_ripe_atlas/processed a partir da pasta atual.")

PASTA_SPLITS = ROOT / "data_ripe_atlas" / "processed"
CAMINHO_TREINO    = PASTA_SPLITS / "treino.csv"
CAMINHO_VALIDACAO = PASTA_SPLITS / "validacao.csv"
CAMINHO_BASELINE  = PASTA_SPLITS / "baseline_por_fluxo.csv"      # o baseline commitado na Tarefa 2
PASTA_MODELOS = ROOT / "data_ripe_atlas" / "models"              # saída: árvore, regras e parâmetros
# O arquivo de teste NÃO é lido neste notebook (fica fechado até a Tarefa 5).

CLASSES = ["OK", "RISCO", "FALHA"]          # ordem de leitura da matriz 3x3
ALVO = "classe"
FEATURES = ["z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
            "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"]
NAO_PODEM_ENTRAR = ["pais_origem", "pais_destino", "rota", "dst_addr", "prb_id", "fluxo_id",
                    "avg", "rtt_min_ms", "rtt_max_ms", "jitter_ms", "mediana", "MAD", "jitter_tipico"]

CRITERIO = "gini"                            # CART com índice de Gini
SEMENTE = 42                                 # random_state (desempate entre divisões equivalentes)
GRADE_PROFUNDIDADE = [2, 3, 4, 5, 6, 7, 8, 10, 12, 15, 20]
GRADE_MIN_FOLHA = [1, 5, 10, 20, 50, 100, 200, 500]
TOLERANCIA_F1 = 0.005                        # aceita até 0,005 abaixo do melhor F1 macro e escolhe a menor árvore

def milhar(n): return f"{int(n):,}".replace(",", ".")
def br(x, casas=2): return f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

## 1. Colunas que a árvore oficial pode usar

Entram: `z_robusto`, `aumento_pct`, `jitter_relativo`, `perda_pct`, `timeout_atual`, `n5_timeout`, `n5_aumento80`, `n5_risco`.
Não entram: país, IP, `rota`, `fluxo_id`, nome do destino, RTT em milissegundos no lugar das métricas relativas.
O alvo é a coluna `classe`; ela **não** é feature.

- [ ] Treino, validação e teste são os da Tarefa 2
- [ ] A ficha de baseline não foi recalculada com o Período B
- [ ] Lista de colunas do treino colada no diário
- [ ] Conferido que a classe é o alvo, não uma feature

In [3]:
tr = pd.read_csv(CAMINHO_TREINO)
va = pd.read_csv(CAMINHO_VALIDACAO)
base = pd.read_csv(CAMINHO_BASELINE)

# 1) São os blocos da Tarefa 2?
print("Treino    N:", milhar(len(tr)), "| conjunto:", sorted(tr["conjunto"].unique()), "| periodo:", sorted(tr["periodo"].unique()))
print("Validação N:", milhar(len(va)), "| conjunto:", sorted(va["conjunto"].unique()), "| periodo:", sorted(va["periodo"].unique()))
assert set(tr["conjunto"]) == {"treino"} and set(va["conjunto"]) == {"validacao"}
assert set(tr["periodo"]) == {"B"} and set(va["periodo"]) == {"B"}, "Há registro do Período A"
lim_tr = tr.groupby("fluxo_id")["timestamp"].max()
lim_va = va.groupby("fluxo_id")["timestamp"].min()
assert (lim_tr < lim_va.reindex(lim_tr.index)).all(), "Treino e validação se misturam em algum fluxo"
print("Em cada fluxo, o treino é anterior à validação: OK")
print("Esperado na Tarefa 2: treino 95.406 e validação 38.138 | obtido:", milhar(len(tr)), "e", milhar(len(va)))

# 2) A ficha é a congelada? (a ficha repetida em cada linha precisa ser a do baseline commitado)
ficha_nas_linhas = pd.concat([tr, va]).groupby("fluxo_id")[["mediana", "MAD", "jitter_tipico"]].agg(["nunique"])
assert (ficha_nas_linhas.to_numpy() == 1).all(), "A ficha muda dentro de um fluxo"
cmp = (pd.concat([tr, va]).drop_duplicates("fluxo_id").set_index("fluxo_id")[["mediana", "MAD", "jitter_tipico"]]
       .join(base.set_index("fluxo_id")[["mediana", "MAD", "jitter_tipico"]], rsuffix="_baseline", how="outer"))
assert cmp.notna().all().all(), "Fluxos diferentes entre o dataset e o baseline"
for c in ("mediana", "MAD", "jitter_tipico"):
    assert np.allclose(cmp[c], cmp[c + "_baseline"], rtol=0, atol=1e-6), f"{c} difere do baseline"
print(f"Ficha das linhas == baseline_por_fluxo.csv ({len(base)} fluxos): OK")

# 3) Colunas
assert ALVO not in FEATURES, "O alvo não pode ser feature"
assert not set(FEATURES) & set(NAO_PODEM_ENTRAR), "Há coluna proibida entre as features"
print()
print("Colunas usadas pela árvore oficial:", ", ".join(FEATURES))
print("Alvo:", ALVO)
print("Colunas do treino que ficam de fora (auditoria ou proibidas):", ", ".join(c for c in tr.columns if c not in FEATURES + [ALVO]))
print()
print("Classes no treino:", tr[ALVO].value_counts().reindex(CLASSES).to_dict())
print("Classes na validação:", va[ALVO].value_counts().reindex(CLASSES).to_dict())
print("Valores vazios nas features (treino):", {c: int(tr[c].isna().sum()) for c in FEATURES if tr[c].isna().any()})

X_tr, y_tr = tr[FEATURES], tr[ALVO]
X_va, y_va = va[FEATURES], va[ALVO]

Treino    N: 95.406 | conjunto: ['treino'] | periodo: ['B']
Validação N: 38.138 | conjunto: ['validacao'] | periodo: ['B']
Em cada fluxo, o treino é anterior à validação: OK
Esperado na Tarefa 2: treino 95.406 e validação 38.138 | obtido: 95.406 e 38.138
Ficha das linhas == baseline_por_fluxo.csv (76 fluxos): OK

Colunas usadas pela árvore oficial: z_robusto, aumento_pct, jitter_relativo, perda_pct, timeout_atual, n5_timeout, n5_aumento80, n5_risco
Alvo: classe
Colunas do treino que ficam de fora (auditoria ou proibidas): fluxo_id, prb_id, dst_addr, measurement_id, timestamp, datetime_utc, avg, rtt_min_ms, rtt_max_ms, sent, rcvd, jitter_ms, pais_origem, pais_destino, rota, periodo, mediana, MAD, jitter_tipico, crit_risco_ind, conjunto

Classes no treino: {'OK': 72857, 'RISCO': 6582, 'FALHA': 15967}
Classes na validação: {'OK': 27180, 'RISCO': 3388, 'FALHA': 7570}
Valores vazios nas features (treino): {'z_robusto': 197, 'aumento_pct': 197, 'jitter_relativo': 261}


## 2. A árvore oficial

CART com critério **Gini**. `max_depth` e `min_samples_leaf` são escolhidos olhando **só a validação**, pelo **F1 macro** (a acurácia sozinha não decide: com maioria OK ela fica alta mesmo errando FALHA). Entre as configurações até `TOLERANCIA_F1` abaixo do melhor F1 macro, fica a de **menos folhas**, porque as regras precisam ser lidas em português. O `fit` é feito **só no treino**.

- [ ] Uma árvore de decisão (CART: Gini)
- [ ] `max_depth` e `min_samples_leaf` escolhidos olhando só a validação
- [ ] Semente registrada
- [ ] Ajuste (`fit`) só no treino
- [ ] Profundidade real, número de folhas e primeiras divisões descritas
- [ ] Pelo menos três regras copiadas da árvore

In [4]:
def nova_arvore(max_depth, min_samples_leaf):
    return DecisionTreeClassifier(criterion=CRITERIO, max_depth=max_depth,
                                  min_samples_leaf=min_samples_leaf, random_state=SEMENTE)

linhas = []
for md_ in GRADE_PROFUNDIDADE:
    for msl in GRADE_MIN_FOLHA:
        arv = nova_arvore(md_, msl).fit(X_tr, y_tr)                      # fit só no treino
        pred = arv.predict(X_va)                                         # avaliação só na validação
        linhas.append({"max_depth": md_, "min_samples_leaf": msl,
                       "f1_macro_val": f1_score(y_va, pred, labels=CLASSES, average="macro", zero_division=0),
                       "acuracia_val": float((pred == y_va).mean()),
                       "profundidade_real": arv.get_depth(), "folhas": arv.get_n_leaves()})
grade = pd.DataFrame(linhas)

melhor = grade["f1_macro_val"].max()
candidatas = grade[grade["f1_macro_val"] >= melhor - TOLERANCIA_F1]
escolhida = candidatas.sort_values(["folhas", "max_depth", "min_samples_leaf"], ascending=[True, True, False]).iloc[0]
MAX_DEPTH = int(escolhida["max_depth"]); MIN_FOLHA = int(escolhida["min_samples_leaf"])

print(f"Melhor F1 macro na validação: {br(melhor, 4)} | tolerância: {br(TOLERANCIA_F1, 3)} | configurações dentro da tolerância: {len(candidatas)}")
print()
print("Melhor F1 macro (validação) para cada profundidade pedida:")
resumo_prof = (grade.sort_values("f1_macro_val", ascending=False).drop_duplicates("max_depth")
               .sort_values("max_depth")[["max_depth", "min_samples_leaf", "f1_macro_val", "profundidade_real", "folhas"]])
resumo_prof["f1_macro_val"] = resumo_prof["f1_macro_val"].round(4)
print(resumo_prof.to_string(index=False))
print()
print(f"ESCOLHIDA: max_depth = {MAX_DEPTH} | min_samples_leaf = {MIN_FOLHA} | F1 macro (validação) = {br(escolhida['f1_macro_val'], 4)} | folhas = {int(escolhida['folhas'])}")

Melhor F1 macro na validação: 0,9999 | tolerância: 0,005 | configurações dentro da tolerância: 54

Melhor F1 macro (validação) para cada profundidade pedida:
 max_depth  min_samples_leaf  f1_macro_val  profundidade_real  folhas
         2                 1        0.6416                  2       3
         3                50        0.9357                  3       5
         4                 5        0.9970                  4       8
         5                 1        0.9980                  5      12
         6                 1        0.9989                  6      15
         7                 1        0.9999                  7      17
         8                 1        0.9997                  8      18
        10                 1        0.9997                  8      18
        12                 1        0.9997                  8      18
        15                 1        0.9997                  8      18
        20                 1        0.9997                  8      18

E

In [5]:
# Árvore oficial: fit só no treino
arvore = nova_arvore(MAX_DEPTH, MIN_FOLHA).fit(X_tr, y_tr)

print("Critério:", CRITERIO, "| semente (random_state):", SEMENTE)
print("Profundidade máxima pedida:", MAX_DEPTH, "| profundidade obtida:", arvore.get_depth(),
      "| folhas:", arvore.get_n_leaves(), "| min_samples_leaf:", MIN_FOLHA)
print("F1 macro no treino (só referência, não decide nada):",
      br(f1_score(y_tr, arvore.predict(X_tr), labels=CLASSES, average="macro", zero_division=0), 4))
print()
print("Importância das features:")
print(pd.Series(arvore.feature_importances_, index=FEATURES).sort_values(ascending=False).round(4).to_string())
print()
texto_arvore = export_text(arvore, feature_names=FEATURES, decimals=2)
print("Árvore (texto):")
print(texto_arvore)

Critério: gini | semente (random_state): 42
Profundidade máxima pedida: 4 | profundidade obtida: 4 | folhas: 8 | min_samples_leaf: 100
F1 macro no treino (só referência, não decide nada): 0,9967

Importância das features:
z_robusto          0.7516
n5_risco           0.1497
jitter_relativo    0.0901
perda_pct          0.0086
n5_aumento80       0.0001
aumento_pct        0.0000
n5_timeout         0.0000
timeout_atual      0.0000

Árvore (texto):
|--- z_robusto <= 3.50
|   |--- n5_risco <= 1.50
|   |   |--- perda_pct <= 16.67
|   |   |   |--- n5_aumento80 <= 0.50
|   |   |   |   |--- class: OK
|   |   |   |--- n5_aumento80 >  0.50
|   |   |   |   |--- class: OK
|   |   |--- perda_pct >  16.67
|   |   |   |--- class: FALHA
|   |--- n5_risco >  1.50
|   |   |--- z_robusto <= 1.99
|   |   |   |--- jitter_relativo <= 3.00
|   |   |   |   |--- class: OK
|   |   |   |--- jitter_relativo >  3.00
|   |   |   |   |--- class: RISCO
|   |   |--- z_robusto >  1.99
|   |   |   |--- z_robusto <= 2.02
| 

In [6]:
INTEIRAS = {"timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"}

def condicao(nome, lado, limiar):
    if nome in INTEIRAS:                                   # limiar 2,50 em contagem inteira vira "<= 2" / ">= 3"
        k = int(np.floor(limiar))
        return f"{nome} <= {k}" if lado == "<=" else f"{nome} >= {k + 1}"
    return f"{nome} {lado} {br(limiar)}"

def regras_das_folhas(clf, nomes):
    t, saida = clf.tree_, []
    def desce(no, conds):
        if t.children_left[no] == -1:
            v = t.value[no][0]; v = v / v.sum(); k = int(np.argmax(v))
            saida.append({"se": " e ".join(conds) if conds else "(sempre)", "classe": clf.classes_[k],
                          "amostras": int(t.n_node_samples[no]), "pureza": float(v[k])})
            return
        nome, lim = nomes[t.feature[no]], float(t.threshold[no])
        desce(t.children_left[no],  conds + [condicao(nome, "<=", lim)])
        desce(t.children_right[no], conds + [condicao(nome, ">", lim)])
    desce(0, [])
    return pd.DataFrame(saida)

regras = regras_das_folhas(arvore, FEATURES)
print(f"{len(regras)} folhas. Maiores folhas de cada classe (regra = caminho da raiz até a folha):")
for c in CLASSES:
    sub = regras[regras["classe"] == c].sort_values("amostras", ascending=False).head(3)
    for r in sub.itertuples():
        print(f"- SE {r.se} ENTÃO {r.classe}   [{milhar(r.amostras)} medições do treino, {br(r.pureza * 100, 1)}% da folha é {r.classe}]")
print()
print("Primeiras divisões:", [(FEATURES[arvore.tree_.feature[n]], round(float(arvore.tree_.threshold[n]), 2)) for n in (0, arvore.tree_.children_left[0], arvore.tree_.children_right[0]) if arvore.tree_.children_left[n] != -1])

8 folhas. Maiores folhas de cada classe (regra = caminho da raiz até a folha):
- SE z_robusto <= 3,50 e n5_risco <= 1 e perda_pct <= 16,67 e n5_aumento80 <= 0 ENTÃO OK   [65.840 medições do treino, 100,0% da folha é OK]
- SE z_robusto <= 3,50 e n5_risco >= 2 e z_robusto <= 1,99 e jitter_relativo <= 3,00 ENTÃO OK   [6.845 medições do treino, 98,4% da folha é OK]
- SE z_robusto <= 3,50 e n5_risco <= 1 e perda_pct <= 16,67 e n5_aumento80 >= 1 ENTÃO OK   [300 medições do treino, 94,3% da folha é OK]
- SE z_robusto <= 3,50 e n5_risco >= 2 e z_robusto > 1,99 e z_robusto > 2,02 ENTÃO RISCO   [4.278 medições do treino, 99,6% da folha é RISCO]
- SE z_robusto <= 3,50 e n5_risco >= 2 e z_robusto <= 1,99 e jitter_relativo > 3,00 ENTÃO RISCO   [2.228 medições do treino, 99,2% da folha é RISCO]
- SE z_robusto <= 3,50 e n5_risco >= 2 e z_robusto > 1,99 e z_robusto <= 2,02 ENTÃO RISCO   [100 medições do treino, 93,0% da folha é RISCO]
- SE z_robusto > 3,50 ENTÃO FALHA   [15.659 medições do treino, 100

## 3. Leitura do erro (validação)

O teste continua fechado. A leitura do erro usa o bloco de **validação**.

In [7]:
pred_va = arvore.predict(X_va)

mat = pd.DataFrame(confusion_matrix(y_va, pred_va, labels=CLASSES),
                   index=[f"real {c}" for c in CLASSES], columns=[f"prevista {c}" for c in CLASSES])
print("Matriz 3x3 (validação) — linhas: classe real; colunas: classe prevista pela árvore")
print(mat.to_string())
print()
p, r, f, s = precision_recall_fscore_support(y_va, pred_va, labels=CLASSES, zero_division=0)
metricas = pd.DataFrame({"precisão": p, "recall": r, "F1": f, "medições": s}, index=CLASSES)
print(metricas.round(4).to_string())
f1_macro = f1_score(y_va, pred_va, labels=CLASSES, average="macro", zero_division=0)
acuracia = float((pred_va == y_va).mean())
print()
print(f"F1 macro (validação): {br(f1_macro, 4)}")
print(f"Acurácia (só contexto): {br(acuracia * 100, 2)}% | proporção de OK na validação: {br((y_va == 'OK').mean() * 100, 2)}% "
      f"(prever sempre OK já daria essa acurácia sem acertar nenhuma FALHA)")

Matriz 3x3 (validação) — linhas: classe real; colunas: classe prevista pela árvore
            prevista OK  prevista RISCO  prevista FALHA
real OK           27172               8               0
real RISCO           11            3377               0
real FALHA           41              12            7517

       precisão  recall      F1  medições
OK       0.9981  0.9997  0.9989     27180
RISCO    0.9941  0.9968  0.9954      3388
FALHA    1.0000  0.9930  0.9965      7570

F1 macro (validação): 0,9969
Acurácia (só contexto): 99,81% | proporção de OK na validação: 71,27% (prever sempre OK já daria essa acurácia sem acertar nenhuma FALHA)


In [8]:
# Erros concretos (validação)
val = va.copy()
val["previsao"] = pred_va
# Só auditoria: qual linha da tabela de rotulagem da Tarefa 2 definiu a classe real (recalculada a partir das colunas do arquivo)
_z = val["z_robusto"].fillna(-np.inf)
val["linha_rotulo"] = np.select(
    [val["perda_pct"] >= 10, val["n5_timeout"] >= 3, _z >= 3.5, val["n5_aumento80"] >= 2,
     (val["crit_risco_ind"] == 1) & (val["n5_risco"] >= 2)], [1, 2, 3, 4, 5], default=6)
cols_erro = ["fluxo_id", "datetime_utc", "mediana", "avg", "z_robusto", "aumento_pct", "jitter_relativo",
             "perda_pct", "n5_timeout", "n5_aumento80", "crit_risco_ind", "n5_risco", "classe", "previsao", "linha_rotulo"]

longo_estavel_em_falha = val[(val["previsao"] == "FALHA") & (val["classe"] != "FALHA") & (val["mediana"] > 150)
                             & (val["z_robusto"].abs() < 2) & (val["perda_pct"] == 0)]
falha_curta_em_ok = val[(val["classe"] == "FALHA") & (val["previsao"] == "OK") & (val["mediana"] < 5)]

print("(a) Caminho longo estável (mediana > 150 ms, |z_robusto| < 2, sem perda) previsto como FALHA:", milhar(len(longo_estavel_em_falha)))
print("(b) FALHA de caminho curto (mediana < 5 ms) prevista como OK:", milhar(len(falha_curta_em_ok)))
for titulo, df_ in (("(a)", longo_estavel_em_falha), ("(b)", falha_curta_em_ok)):
    if len(df_) == 0:
        print(f"\n{titulo}: o caso não apareceu na validação.")
    else:
        print(f"\n{titulo}: primeiros exemplos (colunas mediana e avg são só auditoria):")
        print(df_.sort_values(["fluxo_id", "timestamp"]).head(2)[cols_erro].to_string(index=False))

erros = val[val["classe"] != val["previsao"]]
pares = erros.groupby(["classe", "previsao"]).size().sort_values(ascending=False)
print()
print(f"Erros na validação: {milhar(len(erros))} de {milhar(len(val))} ({br(len(erros) / len(val) * 100, 2)}%)")
print("Pares (classe real, classe prevista) com erro:")
print(pares.to_string())
print()
print("Um erro concreto de cada par (colunas mediana, avg e linha_rotulo são só auditoria):")
for (real, prev), _ in pares.items():
    ex = erros[(erros["classe"] == real) & (erros["previsao"] == prev)].sort_values(["fluxo_id", "timestamp"]).iloc[0]
    print(f"\nreal {real} -> prevista {prev}")
    print(ex[cols_erro].to_string())
print()
print("Fluxos com mais erros:", erros["fluxo_id"].value_counts().head(5).to_dict())
print("Erros em fluxos de caminho curto (mediana < 5 ms):", int((erros["mediana"] < 5).sum()),
      "| de caminho longo (mediana > 150 ms):", int((erros["mediana"] > 150).sum()))
print("Linha da tabela da Tarefa 2 que definiu a classe real nos erros:")
print(pd.crosstab([erros["classe"], erros["previsao"]], erros["linha_rotulo"]).to_string())

(a) Caminho longo estável (mediana > 150 ms, |z_robusto| < 2, sem perda) previsto como FALHA: 0
(b) FALHA de caminho curto (mediana < 5 ms) prevista como OK: 0

(a): o caso não apareceu na validação.

(b): o caso não apareceu na validação.

Erros na validação: 72 de 38.138 (0,19%)
Pares (classe real, classe prevista) com erro:
classe  previsao
FALHA   OK          41
        RISCO       12
RISCO   OK          11
OK      RISCO        8

Um erro concreto de cada par (colunas mediana, avg e linha_rotulo são só auditoria):

real FALHA -> prevista OK
fluxo_id                 6659|103.179.107.33
datetime_utc       2026-09-17T02:20:07+00:00
mediana                           236.597873
avg                               237.092921
z_robusto                           0.266597
aumento_pct                         0.209236
jitter_relativo                     0.045049
perda_pct                          33.333333
n5_timeout                               0.0
n5_aumento80                             0.0

## 4. Árvore de contraste (não é o modelo do projeto)

Outra árvore, **no mesmo treino**, com os mesmos `max_depth`, `min_samples_leaf` e semente, mas **acrescentando** o RTT absoluto (`avg`) ou a rota. Serve só para ver se, com o atalho disponível, ela o usa. **Fica fora da entrega final.** Uma terceira variante, extra, usa **somente** RTT absoluto e rota, para medir o quanto a distância sozinha explica a classe.

- [ ] Dizer qual divisão apareceu perto da raiz
- [ ] Declarar que essa árvore fica fora da entrega final

In [9]:
rotas = sorted(tr["rota"].unique())
def com_rota(d):
    dum = pd.get_dummies(d["rota"]).reindex(columns=rotas, fill_value=0).astype(int)
    dum.columns = [f"rota_{c}" for c in dum.columns]
    return dum

def montar(d, usar_relativas, usar_avg, usar_rota):
    partes = []
    if usar_relativas: partes.append(d[FEATURES])
    if usar_avg: partes.append(d[["avg"]])
    if usar_rota: partes.append(com_rota(d))
    return pd.concat(partes, axis=1)

variantes = [("C1: métricas relativas + RTT absoluto (avg)", True, True, False),
             ("C2: métricas relativas + rota", True, False, True),
             ("C3 (extra): só RTT absoluto + rota", False, True, True)]

for nome, rel, av, ro in variantes:
    Xt, Xv = montar(tr, rel, av, ro), montar(va, rel, av, ro)
    cont = nova_arvore(MAX_DEPTH, MIN_FOLHA).fit(Xt, y_tr)
    t = cont.tree_
    raiz = f"{Xt.columns[t.feature[0]]} <= {br(float(t.threshold[0]))}"
    imp = pd.Series(cont.feature_importances_, index=Xt.columns)
    atalho = imp[[c for c in Xt.columns if c == "avg" or c.startswith("rota_")]].sum()
    f1c = f1_score(y_va, cont.predict(Xv), labels=CLASSES, average="macro", zero_division=0)
    print("=" * 78)
    print(nome)
    print("Divisão na raiz:", raiz, "| profundidade:", cont.get_depth(), "| folhas:", cont.get_n_leaves())
    print(f"Importância somada de RTT absoluto/rota: {br(atalho * 100, 1)}% | F1 macro na validação (só contexto): {br(f1c, 4)}")
    print("Duas primeiras camadas:")
    print(export_text(cont, feature_names=list(Xt.columns), decimals=2, max_depth=2))
print("Referência — árvore oficial: divisão na raiz:", f"{FEATURES[arvore.tree_.feature[0]]} <= {br(float(arvore.tree_.threshold[0]))}",
      "| F1 macro (validação):", br(f1_macro, 4))
print("Estas árvores de contraste NÃO fazem parte da entrega final.")

C1: métricas relativas + RTT absoluto (avg)
Divisão na raiz: z_robusto <= 3,50 | profundidade: 4 | folhas: 8
Importância somada de RTT absoluto/rota: 0,0% | F1 macro na validação (só contexto): 0,9969
Duas primeiras camadas:
|--- z_robusto <= 3.50
|   |--- n5_risco <= 1.50
|   |   |--- perda_pct <= 16.67
|   |   |   |--- truncated branch of depth 2
|   |   |--- perda_pct >  16.67
|   |   |   |--- class: FALHA
|   |--- n5_risco >  1.50
|   |   |--- z_robusto <= 1.99
|   |   |   |--- truncated branch of depth 2
|   |   |--- z_robusto >  1.99
|   |   |   |--- truncated branch of depth 2
|--- z_robusto >  3.50
|   |--- class: FALHA

C2: métricas relativas + rota
Divisão na raiz: z_robusto <= 3,50 | profundidade: 4 | folhas: 8
Importância somada de RTT absoluto/rota: 0,0% | F1 macro na validação (só contexto): 0,9969
Duas primeiras camadas:
|--- z_robusto <= 3.50
|   |--- n5_risco <= 1.50
|   |   |--- perda_pct <= 16.67
|   |   |   |--- truncated branch of depth 2
|   |   |--- perda_pct >  

## 5. Salvar a árvore oficial (para a Tarefa 4)

In [10]:
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
params = {"criterio": CRITERIO, "max_depth_pedido": MAX_DEPTH, "min_samples_leaf": MIN_FOLHA,
          "random_state": SEMENTE, "profundidade_obtida": int(arvore.get_depth()), "folhas": int(arvore.get_n_leaves()),
          "features": FEATURES, "alvo": ALVO, "treinada_em": "treino.csv", "escolha_de_hiperparametros": "validacao.csv (F1 macro, tolerancia %.3f)" % TOLERANCIA_F1,
          "f1_macro_validacao": round(float(f1_macro), 6), "scikit_learn": sklearn.__version__}
(PASTA_MODELOS / "arvore_decisao_T3_params.json").write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
(PASTA_MODELOS / "arvore_decisao_T3_texto.txt").write_text(texto_arvore, encoding="utf-8")
joblib.dump(arvore, PASTA_MODELOS / "arvore_decisao_T3.joblib")
print("Salvo em", PASTA_MODELOS, ":", ", ".join(sorted(p.name for p in PASTA_MODELOS.iterdir())))

Salvo em C:\ProjetosVScode\Projeto4Semestre\Modelo-preditivo---Projeto-Acad-mico---4-Semestre\data_ripe_atlas\models : arvore_decisao_T3.joblib, arvore_decisao_T3_params.json, arvore_decisao_T3_texto.txt, arvore_oficial.joblib, arvore_oficial_params.json, arvore_oficial_texto.txt, floresta_final.joblib, floresta_final_params.json, floresta_final_regras.txt, resultado_fluxos_novos.json, resultado_teste.json, teste_medido.json


## Checklist (marque no diário depois de conferir as saídas)

- [ ] Treino, validação e teste são os da Tarefa 2 (seção 1)
- [ ] A ficha de baseline não foi recalculada com o Período B (seção 1)
- [ ] Lista de colunas e alvo conferidos (seção 1)
- [ ] Gini, profundidade, `min_samples_leaf` e semente declarados; escolha pela validação (seção 2)
- [ ] Profundidade real, folhas e três regras (seção 2)
- [ ] Matriz 3×3, precisão/recall/F1 por classe, F1 macro, dois erros concretos (seção 3)
- [ ] Árvore de contraste: divisão na raiz e declaração de que fica fora da entrega (seção 4)
- [ ] Board atualizado e tabela de integrantes (seção 5 do diário)